

<div style="max-width: 850px;">

# **LLM - The Transformer lifecycle**

---

This project is used to understand the lifecycle of transformers 
From data cleaning to speed up inference
We will keep it small such that we can train several transformers
All will be implemented from scratch
Later we can upgrade to use everything using hugging face interfaces


**Requirements**

These are requirements and how to install them in your environment

* `pip install datasets`

</div>



<div style="max-width: 850px;">

# The dataset

---

## Data curation and preparation

---

We will start small, using `tiny stories` dataset.
This dataset lives in huggingface. 

We will observe it's distribution and compute useful statistics from the dataset.

To process text we should also use a tokenizer. At the moment we are not interested in training one ourselves and we will use the `GPT2TokenizerFast` available in Huggingface. `GPT2TokenizeFast` implements a BPE (byte-pair encoding) tokenization.

</div>


In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import tqdm
import logging

import sys
from pathlib import Path

import numpy as np
from wordcloud import WordCloud


from transformers import GPT2TokenizerFast

# To avoid getting blocked by sending many requests
from huggingface_hub import notebook_login

# imports from transformers lyfecycle

# Support kernels launched from the project root or notebooks directory.
project_root = Path.cwd().resolve()
if not (project_root / 'src' / 'data_cycle.py').is_file():
    project_root = project_root.parent
if not (project_root / 'src' / 'data_cycle.py').is_file():
    raise FileNotFoundError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(project_root / 'src'))

import data_cycle

notebook_login()

logging.basicConfig(level=logging.INFO) 
_LOGGER = logging.getLogger(__name__)

data_path = project_root / 'datasets'

# Experiment configuration and reusable processors.
train_source = data_cycle.TINYSTORIES_TRAIN_SOURCE
valid_source = data_cycle.TINYSTORIES_VALIDATION_SOURCE
code_source = data_cycle.PYTHON_CODE_TRAIN_SOURCE

num_proc = 4
preprocess_batch_size = 1000
tokenization_batch_size = 512
signature_batch_size = 256

normalization_config = data_cycle.NormalizationConfig(max_new_lines=3)
minhash_config = data_cycle.MinHashConfig(
    lsh_threshold=0.85,
    similarity_threshold=0.85,
    verify_similarity=True,
)
token_display_config = data_cycle.DEFAULT_TOKEN_DISPLAY_CONFIG

train_normalizer = data_cycle.TextNormalizer(
    transformations=data_cycle.PROSE_NORMALIZATION,
    config=normalization_config,
    text_column=train_source.text_column,
)
valid_normalizer = data_cycle.TextNormalizer(
    transformations=data_cycle.PROSE_NORMALIZATION,
    config=normalization_config,
    text_column=valid_source.text_column,
)
train_deduplicator = data_cycle.MinHashDeduplicator(
    config=minhash_config, text_column=train_source.text_column,
)
valid_deduplicator = data_cycle.MinHashDeduplicator(
    config=minhash_config, text_column=valid_source.text_column,
)


In [ ]:
# Load each configured split without changing its columns.
stories_train_dataset = data_cycle.load_source(train_source)
stories_valid_dataset = data_cycle.load_source(valid_source)


In [ ]:
# display dataset information and a few examples
data_cycle.display_dataset_information(stories_train_dataset)
print(stories_train_dataset.select(range(10)).to_pandas())



<div style="max-width: 850px;">

## Dataset diagnosis
---

We run data diagnosis by computing and observing dataset statistics. These will help us to decide what pre-processing steps we need to apply to clean, standrize and filter documents. 

Usually this step is performed on a tokenized dataset. We will keep it simple and use `GPT2TokenizerFast`, which is a tokenizer trained using the BytePairing algorithm, and compute the following:

   - **Number of documents**: a document can be simply put as the basic unit in the dataset. This is a story within `tiny stories` dataset.
   - **Document length distribution**: the distribution help us identify documents that are too short or too long. 
   - **Vocabulary/Token distribution**: we will plot Zipf plots. These help us to identify if there are tokens that are too often repeating - such as html tags - and can indicate garbage or spurious dataset. For more context on these plots look [at this article.](https://www.nngroup.com/articles/zipf-curves-and-website-popularity/#:~:text=According%20to%20much%20available%20data%2C%20web%20use,number%20of%20elements%20that%20score%20very%20low)

**NOTE** `GPT2TokenizerFast` has a maximun length of 1024. Any document with longer length will be cliped by it.

</div>



In [ ]:
# Tokenizing dataset

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# apply a function to map and pre process the data to tokens
# for tokenizing the text and generate a sequence of tokens
def tokenize_preprocess_fn(row_example):
    return {
        'text': row_example['text'], 
        'tokenized': tokenizer(row_example['text'])['input_ids']
    }

stories_train_dataset = stories_train_dataset.map(
    tokenize_preprocess_fn,
    batched=True,
    batch_size=preprocess_batch_size,
    num_proc=num_proc,
    desc="Tokenizing text"
)


In [ ]:
# computing disrtibution of length and tokens

# vocabulary
vocabulary = tokenizer.get_vocab()

_LOGGER.info(f'Len of vocabulary: {len(vocabulary)}')
the_lengths = [
    len(dd['tokenized']) 
    for dd in tqdm.tqdm(stories_train_dataset, desc='Get lengths of data')
]

the_ids = []
for d in tqdm.tqdm(stories_train_dataset, desc='Gathering tokens'):
    the_ids.extend(d['tokenized'])

id_dist = {x:0 for x in set(the_ids)}
for k in tqdm.tqdm(the_ids, desc='Computing token distribution'):
    id_dist[k] += 1

id_dist_list = sorted([(k, v) for k, v in id_dist.items()], key=lambda x: x[0])


In [ ]:
# Looking to some parts of the dataset

# Let's look at the largest ocurrence. We will find that there is only a few
# tokens that are taking a lot of the counts -> over 10M occurences!
topk = 10
id_dist__ = sorted([(k, v) for (k, v) in id_dist_list], key=lambda x: x[1])

_LOGGER.info(f' The following are the {topk} tokens with largest ocurrence')
vocab_reversed = {v:k for k, v in vocabulary.items()}

for k, v in [[vocab_reversed[x[0]], x[1]] for x in id_dist__[-topk:]]:
    _LOGGER.info(f' {k} -> {v}')


In [ ]:
# plot counts and world cloud of tokens


mean, median, std = np.mean(the_lengths), np.median(the_lengths), np.std(the_lengths)

_LOGGER.info(f' Total of documents in the dataset: {len(stories_train_dataset)}')
_LOGGER.info(f' Document length statistics: mean:{mean}; median: {median}; std: {std}')
_LOGGER.info(f' Vocabulary size of tokenizer: {len(vocabulary)}')


fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0,0].hist(the_lengths, bins=50, color='slateblue', edgecolor='white')
axes[0,0].axvline(x=mean, linestyle='--', color='orange', label='mean')
axes[0,0].axvline(x=median, linestyle='--', color='red', label='median')

axes[0,0].axvline(x=mean+2*std, linestyle='--', color='green', label='mean+/- 2 std')
axes[0,0].axvline(x=mean-2*std, linestyle='--', color='green', label='mean+/- 2 std')

axes[0,0].legend()
axes[0,0].set_xlabel("Length of sequence (tokenized)")
axes[0,0].set_title('Histogram - document length')
axes[0,0].set_ylabel('Counts')
axes[0,0].grid(True, linestyle='--')

token_counts = data_cycle.ids_to_token_counts(id_dist, tokenizer, config=token_display_config)
max_words = 200

wc = WordCloud(
    width=1000,
    height=1000,
    background_color="white",
    max_words=max_words
)

wc.generate_from_frequencies(token_counts)
axes[0,1].imshow(wc)
axes[0,1].axis("off")
axes[0,1].set_title("Token Frequency (raw)")

counts = np.array(sorted(id_dist.values(), reverse=True))
ranks = np.arange(1, len(counts) + 1)
# ideal Zipf curve (scaled to match first point)
ideal = counts[0] / ranks

axes[1,0].loglog(ranks, counts, label='Empirical')
axes[1,0].loglog(ranks, ideal, linestyle="--", label="Ideal Zipf (1/r)")
axes[1,0].set_xlabel("Token rank (log)")
axes[1,0].set_ylabel("Token count (log)")
axes[1,0].set_title("Token Frequency Distribution / Zipf Plot")
axes[1,0].grid(True)
axes[1,0].legend()


display_counts = data_cycle.filter_token_counts_for_display(id_dist, tokenizer, config=token_display_config)
wc.generate_from_frequencies(display_counts)
axes[1,1].imshow(wc)
axes[1,1].axis("off")
axes[1,1].set_title("Token Frequency (no-punct)")



In [ ]:
# Observing the most and least ranked tokens
N = 10  # configurable

# Sort (token_id, count) by frequency
ranked_tokens = sorted(
    id_dist.items(),
    key=lambda x: x[1],
    reverse=True
)

def show_token(tokenizer, token_id):
    # Raw GPT-2 token representation + decoded human-readable text
    raw = tokenizer.convert_ids_to_tokens(token_id)
    decoded = tokenizer.decode([token_id])
    return raw, decoded


print(f"Top {N} highest-ranked tokens:")
for rank, (token_id, count) in enumerate(ranked_tokens[:N], start=1):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )

print(f"\nBottom {N} lowest-ranked tokens:")
total = len(ranked_tokens)

for rank, (token_id, count) in enumerate(
    ranked_tokens[-N:],
    start=total - N + 1
):
    raw, decoded = show_token(tokenizer, token_id)
    print(
        f"Rank {rank:>6} | "
        f"ID {token_id:>6} | "
        f"token={repr(raw):<18} | "
        f"text={repr(decoded):<18} | "
        f"count={count}"
    )


<div style="max-width: 850px;">

**Diagnosis**


* **Most frequent tokens**. These are punctuation tokens and lots of short length words. For training, keep punctuation and frequent tokens - **removing them breaks language modeling because the model must learn *syntax, spacing, sentence boundaries, formatting, etc.***. **It only becomes a problem** if the top tokens are things like `<, >, +, http, www, etc` or repeated boilerplate tokens from HTML, templates, logs, menus, tables, or corrupted text.

    * **Conclusion**. There are no unwanted tokens in this dataset. When removed punctuation and `articles` such as "the", "that" etc, most frequent tokens become pronouns like "he", "she" - which is normal in stories.


* **Length of documents have long tails**. We don't want documents on both tails - very short documents and very long documents. The length of the documents do not make them "be of bad quality", but it's more of a question of efficient training. For example, very short documents make training inefficient because **the model sees little context**. Short examples often include: titles, menu labels, broken web fragments, captions, etc. Very long documents will simply be cut by a `MAX_SEQUENCE_LENGTH` - loading them into memory will be inefficient. In practice document length will be kept in boundaries `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH` with boundaries depending on the dataset.

    * **Conclusion**. There is a long tail for long documents, but overall the distribution looks good. During our exercise we will keep the length of ducuent short with `MIN_SEQUENCE_LENGTH=128` and `MAX_SEQUENCE_LENGTH=512`.


* **Zipf curves**. Zipf curves help us visualizaing the distributions of the tokens in the dataset by ranking: most frequent token has rank 1. This plot should be looking smooth and decreasing and spikes and clifts on the tail. We can diagnose the curve by visualizaing the ideal Zipf distribution `1/rank`.

    * **Conclusion**. The token-frequency distribution approximately follows a Zipf-like distribution, with token frequency decreasing roughly as a power law with rank. The highest-frequency tokens include punctuation, whitespace-related tokens, and common lexical/subword tokens, which explains the relatively flat head of the distribution. The middle portion of the distribution follows the 1/rank reference reasonably well. At high ranks, the empirical distribution falls below the ideal Zipf curve, forming a long tail of rare tokens. Inspection of these tokens shows meaningful low-frequency terms such as "Bigfoot" and "Iodine", suggesting that the tail contains legitimate story-specific vocabulary rather than simply noise. Therefore, rare tokens should not be removed solely based on their frequency.


</div>



<div style="max-width: 850px;">

## Pre-processing and data curation
---

**Data curation is the first and upmost component on quality control on LLMs**. Curation is performed as an iterative process where apply filters, models and heuristics to improve the overall quality. Common pre-processing steps are removing exact duplicates, removing semantic similar elements, and filter elements based on "explainability". Each step in the data curation cycle contributes in different ways to efficient learning of LLMs. 

Data curation follows a simple recipe:

1. **Normalise text**: unicode conversion, boilerplate, stripping, etc,
3. **Exact deduplication**: using hasing,
4. **Near deduplication**: using minhash and local sensitive hashing (LSH),
5. **Apply tokenization**, we can use the same BPE tokenizer,
6. **Filter by token length**: after tokenization we remove elements outside of `MIN_SEQUENCE_LENGTH` and `MAX_SEQUENCE_LENGTH`,
7. **Perplexity filtering**: using document "explainability" from a LLM as a meassure of the quality (can be a strong or a small LLM),
8. **[optional] Semantic deduplication**: converting documents to embeddings and using nearest neigbours,
9. **Iteratively curate** dataset by training and analysing outputs.


In the following we will implement each of these steps in our dataset.

</div>




<div style="max-width: 850px;">

### Step 1 - Normalizing text

---

In the normalization step we simply convert all the text to unicode, strip and remove more than 1 spaces. Specifically, we will focus in the following normalization

1. **Unicode normalization**. Normalize different unicode representations into a canonical form: curly quotes → standard quotes, accented characters → consistent encoding,
2. **Whitespace normalization**. Collapse multiple spaces, normalize tabs/newlines, strip leading/trailing spaces,
3. **Line break normalization**. Convert `\r\n`, `\r → \n` and collapse excessive newlines (e.g., >2 → 2),
4. **Control character removal**. Remove: non-printable characters, zero-width spaces and Unicode artifacts,
5. **Standardize quotes and punctuation**. “ ” → ", ‘ ’ → ', normalize dashes (— → -),
6. **Lowercasing reduces vocabulary entropy, but removes signal (e.g., named entities)** so it's better to keep casing.

We will apply these normalization steps on the hugging face dataset


</div>


In [ ]:
_LOGGER.info('Normalizing text in training data')
stories_train_dataset = stories_train_dataset.map(
    train_normalizer.normalize_batch,
    batched=True,
    batch_size=preprocess_batch_size,
    num_proc=num_proc,   # adjust based on CPU cores
    desc="Normalizing text"
)

_LOGGER.info('Normalizing text in validation data')
stories_valid_dataset = stories_valid_dataset.map(
    valid_normalizer.normalize_batch,
    batched=True,
    batch_size=preprocess_batch_size,
    num_proc=num_proc,   # adjust based on CPU cores
    desc="Normalizing text"
)


In [ ]:
# Save to disk to later load
# stories_train_dataset.save_to_disk(Path(data_path) / 'tinystories_train_dataset')
# stories_valid_dataset.save_to_disk(Path(data_path) / 'tinystories_val_dataset')



<div style="max-width: 850px;">


### Step 2 -Exact deduplication with hashing

---

We will use a hash to remove exact duplicates. First, we will convert all the text to a hash - after normalization - using `hashlib.sha256`. This will create a set of hashes whifh to perform deduplication and make fast lookups.

</div>


In [ ]:

stories_train_dataset = stories_train_dataset.map(
    data_cycle.add_text_hash,
    fn_kwargs={"text_column": train_source.text_column},
    batched=True,
    batch_size=preprocess_batch_size,
    num_proc=num_proc,
    desc="Train:Hashing normalized text",
)

stories_valid_dataset = stories_valid_dataset.map(
    data_cycle.add_text_hash,
    fn_kwargs={"text_column": valid_source.text_column},
    batched=True,
    batch_size=preprocess_batch_size,
    num_proc=num_proc,
    desc="Val:Hashing normalized text",
)


_LOGGER.info('Removing exact dedups in training dataset')
prev_n = len(stories_train_dataset)
stories_train_dataset = data_cycle.remove_exact_dedups_in_dataset(stories_train_dataset)
_LOGGER.info(f'Train: Removed: {(1-len(stories_train_dataset)/ prev_n)*100} % -> {len(stories_train_dataset)} vs {prev_n}')

_LOGGER.info('Removing exact dedups in validation dataset')
prev_n = len(stories_valid_dataset)
stories_valid_dataset = data_cycle.remove_exact_dedups_in_dataset(stories_valid_dataset)
_LOGGER.info(f'Val: Removed: {(1-len(stories_valid_dataset)/prev_n)*100} % -> {len(stories_valid_dataset)} vs {prev_n}')

                      


<div style="max-width: 850px;">


### Step 3 - Near deduplication

---

Finding near duplicates help us to remove documents in the dataset that are not exact duplicates based on text, but instead have similar structure and content. We can define that two documents are near duplicates if they contain similar local patterns in their `n-grams` that are usually constructed in word level. 

N-grams have the following properties:

1. Preserve local structure,
2. Are robust to small details,
3. Ignore global ordering,
4. Are sensitive to similar/copied structure.

Using the N-grams of two documents, the similarity becomes comparing how much overlap have the two sets:

$$
J(A,B) = \frac{|A\cap B|}{|A\cup B|}.
$$

The above is the *Jaccard similarity*. However, at scale this becomes infeasible given that the complexity is $O(N^2)$, $N$ equal to the length of both sets. The solution is to use **MinHashing**. MinHashing converts the documents to a signature and allows us to compute the above expression with the property:

$$
P(\text{MinHash}(A)=\text{MinHash}(B)) = J(A,B).
$$

**The intuition** behind is that, for a permutation of an n-gram that is in $A\cup B$ it must be in: 1) in $A\cap B$, or 2) must be in A or B. The probability that lies in the intersection is $\frac{|A\cap B|}{|A\cup B|}$.

To implement **MinHashing** we simulate permutations with hash functions $h_i$ to create a document signature:

$$
m_i = h_i(g), g\in A.
$$

Then

$$
\text{signature}(A) = [m_1, m_2, m_3, ..., m_K],
$$

for $k=K$ hash functions. Then we compare two documents by computing:

$$
\hat{J}(A, B) = \frac{1}{K} \sum_{i=1}^K I[m_i(A)=m_i(B)].
$$

However, this computation is still $O(N^2)$. To speed up computation we turn our eyes to LSH (local sensitive hashing) to **help us to only compute the expensive comparison metric on documents that are similar in some bands**. Specifically, LSH performs the following steps:

1. Split each MinHash signature into $b$ bands, each containing $r$ rows:

$$
n = b\cdot r,
$$

where \(n\) is the signature length.

2. Hash each band independently into a hash table/bucket.

3. Define candidate pairs: two documents $A,B$ become a candidate pair if their signatures are identical in all $r$ rows of at least one band.

4 Only for these candidate pairs, compute a more precise similarity measure, such as $\hat{J}(A, B)$.


For example, for a two component band, a signature can become:
   
$$
\text{signature}(A) = [[m_1, m_2], [m_3, m_4], ..., [m_{K-1}, m_K]].
$$

Luckily, there are packages in Python that can help us computing these permutations - MinHashes and similarities with LSH. In this exercise we will use `datasketch`. In practice, we need to tune the following parameters:

- **`NGRAM size` — controls the representation.** We split each document into overlapping $n$-grams before computing MinHash. Smaller $n$-grams create more overlap between documents and are more tolerant of small textual changes, but can make unrelated documents appear similar. Larger $n$-grams capture more specific context and reduce accidental matches, but become more sensitive to small edits. **Trade-off:** smaller $n$ → higher recall / lower precision; larger $n$ → higher precision / lower recall.

- **`Number of permutations` — controls the accuracy of MinHash.** This determines the number of hash functions used to construct each MinHash signature. More permutations give a more accurate estimate of the true Jaccard similarity and reduce the variance of $\hat J(A,B)$, but increase hashing cost, memory usage, and LSH index size. **Trade-off:** more permutations → better similarity estimates but more compute and memory; fewer → faster and cheaper but noisier estimates.

- **`Similitud threshold` — controls which similarities LSH targets.** This tells LSH approximately where we want the transition between unlikely and likely candidate pairs to occur. A higher threshold retrieves fewer, more strongly similar candidates but risks missing useful moderately similar pairs. A lower threshold generates more candidates, improving recall but increasing false positives and downstream comparisons. **Trade-off:** higher threshold → higher precision / lower recall; lower threshold → higher recall / lower precision.

- **\(b\) and \(r\) — control the LSH banding behavior.** The MinHash signature is divided into \(b\) bands containing $r$ rows each, roughly satisfying $br=\text{NUM\_PERM}$. Two documents become candidates when they match on all \(r\) rows of at least one band. More bands with fewer rows make LSH more permissive; fewer bands with more rows make it stricter. **Trade-off:** more bands → higher recall / more false positives; more rows per band → higher precision / more false negatives.


</div>


In [ ]:
    
_LOGGER.info('Removing LSH dedups in training dataset')
train_keep, train_duplicates = train_deduplicator.deduplicate(
    stories_train_dataset, num_proc=num_proc, batch_size=signature_batch_size
)
prev_n = len(stories_train_dataset)
stories_train_dataset = stories_train_dataset.select(train_keep)

In [ ]:
_LOGGER.info(f'Train: Removed: {(1-len(stories_train_dataset)/ prev_n)*100} % -> {len(stories_train_dataset)} vs {prev_n}')


In [ ]:

_LOGGER.info('Removing LSH dedups in validation dataset')
prev_n = len(stories_valid_dataset)
val_keep, val_duplicates = valid_deduplicator.deduplicate(
    stories_valid_dataset, num_proc=num_proc, batch_size=signature_batch_size
)
stories_valid_dataset = stories_valid_dataset.select(val_keep)

_LOGGER.info(f'Val: Removed: {(1-len(stories_valid_dataset)/ prev_n)*100} % -> {len(stories_valid_dataset)} vs {prev_n}')




<div style="max-width: 850px;">

### Step 4 - Tokenization

---

A dataset consist of documents. However, dataset is usually regarded as amount of tokens it cointains. Even more, when training a model we usually refer to the total count of tokens the model saw during training considering epochs too - e.g. GPT4 was trained on XX tokens. GPT models use a **Byte-Pair Encoding (BPE)** tokenizers, which decompose words into sub-words based on the most frequent pair of characters in subwords.

As an example, consider the input dataset `'low', 'lower', 'lowest'`. BPE works as follows

1. **Initialize vocabulary** - start with all unique characters as tokens including a token for unknown:

Vocabulary: `'l', 'o', 'w', 'e', 'r', 's', 't', '<unk>'`

2. **Count token pairs in the dataset**

```
(l,o) = 3    (w,e) = 2   (e,s) = 1
(o,w) = 3   (e,r) = 2   (s,t) = 1
```

3. **Merge most frequent pair into new subword token**:

Vocabulary after merging `(l,o)`: `'lo', 'w', 'e', 'r', 's', 't', '<unk>'`

4. **Repeat the process until desired granularity**:

Final vocabulary : `'low', 'lower', 'est', '<unk>'`

5. **The dictionary of vocabulary comprise of**:

`{'low':0, 'lower':1, 'est':3, '<unk>':3}`


We will apply a BPE tokenizer to our training and validation data. Specifically, we will use the tokenizer used for GPT2 models. Later we will compare the statisctics from the clean data w.r.t. the raw dataset.

</div>


In [ ]:

def preprocess_tokenizer_update(batch):
    return {'tokenized': tokenizer(batch['text'])['input_ids']}


_LOGGER.info('Tokenizing training dataset')
stories_train_dataset = stories_train_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=tokenization_batch_size,
    num_proc=num_proc,
    desc='Tokenizing train data'
)

_LOGGER.info('Tokenizing validation dataset')
stories_valid_dataset = stories_valid_dataset.map(
    preprocess_tokenizer_update,
    batched=True,
    batch_size=tokenization_batch_size,
    num_proc=num_proc,
    desc='Tokenizing val data'
)



## Save prepared data

Continue with scoring and filtering in [perplexity_filtering.ipynb](perplexity_filtering.ipynb).


In [ ]:
# Save after normalization, deduplication, and GPT-2 tokenization.
data_path.mkdir(parents=True, exist_ok=True)
stories_train_dataset.save_to_disk(data_path / 'tinystories_prepared_train')
stories_valid_dataset.save_to_disk(data_path / 'tinystories_prepared_validation')